# Laboratório 1: Ingestão de Conhecimento e Criação do Banco Vetorial

**Objetivo deste script:** Ensinar o nosso sistema de Inteligência Artificial a ler um manual técnico humano (PDF). A IA não consegue ler um PDF inteiro de uma vez como nós fazemos. Por isso, este código pega o documento, recorta em pequenos parágrafos com sentido (fatiamento), traduz esses parágrafos para coordenadas matemáticas (embeddings) e guarda tudo em uma "biblioteca" especial chamada Banco de Dados Vetorial (ChromaDB).

In [1]:
# ==========================================
# PASSO 1: IMPORTAÇÃO DAS FERRAMENTAS
# ==========================================
import os
import warnings

# Limpa mensagens de aviso do terminal para não poluir a tela
warnings.filterwarnings("ignore")

# PyMuPDFLoader: É o nosso "leitor de PDF". Ele abre o arquivo e extrai o texto página por página.
from langchain_community.document_loaders import PyMuPDFLoader

# RecursiveCharacterTextSplitter: É a nossa "tesoura inteligente". Ele corta o texto em blocos menores sem quebrar as frases no meio.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# HuggingFaceEmbeddings: É o nosso "tradutor". Transforma texto humano em uma sequência de números que a IA entende.
from langchain_huggingface import HuggingFaceEmbeddings

# Chroma: É a nossa "estante de livros". Onde vamos guardar as sequências de números para buscar depois.
from langchain_chroma import Chroma

### Definindo os Caminhos dos Arquivos
Como este notebook está rodando dentro da pasta `notebooks/`, precisamos avisar o sistema para voltar uma pasta para trás (`../`) e então entrar na pasta `data/` para encontrar nossos manuais e salvar o banco de dados.

In [2]:
# ==========================================
# PASSO 2: CONFIGURAÇÃO DE PASTAS
# ==========================================

# Caminho de onde o PDF está vindo (A matéria-prima)
PDF_PATH = "../data/raw/manual_marton_absurdo.pdf"

# Caminho para onde o Banco Vetorial vai (O produto final)
DB_PATH = "../data/vectordb"

print(f"Buscando o arquivo em: {PDF_PATH}")

Buscando o arquivo em: ../data/raw/manual_marton_absurdo.pdf


### Carregando e Fatiando o Documento (Chunking)
Se você der um livro de 500 páginas para uma pessoa decorar de uma vez, ela vai esquecer os detalhes. Com a IA é a mesma coisa (além de custar muita memória de processamento). 

Aqui usamos a técnica de **Chunking** (Fatiamento). Cortamos o manual em pedaços de 1000 caracteres. Para não perdermos o contexto de um parágrafo para o outro, criamos uma sobreposição (*overlap*) de 150 caracteres. É como se a última frase da página 1 também fosse a primeira frase da página 2, garantindo que o raciocínio não se quebre.

In [3]:
# ==========================================
# PASSO 3: LEITURA E CORTE DO MANUAL
# ==========================================

# 1. Abre e lê o arquivo PDF inteiro
print("Lendo as páginas do PDF...")
loader = PyMuPDFLoader(PDF_PATH)
documento_inteiro = loader.load()

# 2. Configura a regra da tesoura (1000 letras por bloco, 150 de margem de segurança)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

# 3. Executa o corte
textos_fatiados = text_splitter.split_documents(documento_inteiro)

print(f"O manual foi lido e dividido em {len(textos_fatiados)} pequenos blocos de texto.")

Lendo as páginas do PDF...


ValueError: File path ../data/raw/manual_marton_absurdo.pdf is not a valid file or url

### Embeddings e Banco Vetorial (ChromaDB)
Aqui acontece a magia. Modelos de IA não entendem letras, só entendem matemática. 
Nós passamos cada um dos blocos cortados acima pelo modelo `all-MiniLM-L6-v2`. Ele vai ler o texto e gerar **Embeddings** (uma lista gigante de números). 

Textos com assuntos parecidos terão números parecidos. Textos diferentes terão números distantes.
Finalmente, guardamos esses números no **ChromaDB**, que é um banco de dados feito sob medida para pesquisar essas coordenadas matemáticas na velocidade da luz.

In [ ]:
# ==========================================
# PASSO 4: TRADUÇÃO MATEMÁTICA E SALVAMENTO
# ==========================================

# 1. Baixa/Inicializa o tradutor matemático da HuggingFace
print("Iniciando o modelo matemático de Embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Transforma os textos cortados em números e guarda no banco ChromaDB
print("Construindo o Banco Vetorial. Isso pode levar alguns segundos...")
vectordb = Chroma.from_documents(
    documents=textos_fatiados,
    embedding=embeddings,
    persist_directory=DB_PATH # Salva o resultado fisicamente na pasta definida
)

print(f"✅ Ingestão finalizada com sucesso! O cérebro da IA foi guardado na pasta: {DB_PATH}")